In [63]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [64]:
df=pd.read_csv("IMDB.csv")

In [65]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [66]:
df.drop_duplicates(inplace=True)

In [67]:
df.shape

(49582, 2)

Preprocessing

## 1. Converting to Lowercase

In [68]:
df['review']=df['review'].str.lower()

## 2. Removing the URLs

In [69]:
def remove_url(text):
    text=re.sub(r"http\S+" ,"",text)
    return text

df['review']=df['review'].apply(remove_url)

## Removing Punctuations

In [70]:
def remove_punch(text):
    text=re.sub(r"[^A-Za-z0-9\s]" , "", text)
    return text
df["review"]=df["review"].apply(remove_punch)

## 3 Removing HTML

In [71]:
def remove_html(text):
    text=re.sub(r"<.*?>", "", text)
    return text
df["review"]=df["review"].apply(remove_html)    

## 4. Removing the Stopwords

In [72]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word, "")
    return text
df['review']=df['review'].apply(remove_stopwords)


In [73]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### Stemming

In [74]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens= word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)

Encoding

In [75]:
le= LabelEncoder() 

df['sentiment'] = le.fit_transform(df["sentiment"])

In [76]:
y=df['sentiment']
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### Vectorization

In [77]:
tf=TfidfVectorizer(max_features=5000)
X= tf.fit_transform(df['review'])


In [78]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057161 stored elements and shape (49582, 5000)>

Dataset & Data Loaders

In [79]:
X_train, X_test, Y_train, Y_test= train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [80]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [81]:
train_set= TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train.values).float()
)

test_set= TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(Y_test.values).float()
)

In [82]:
train_loader= DataLoader(train_set,shuffle=True, batch_size=64)
test_loader= DataLoader(test_set, shuffle=True, batch_size=64)

### Build RNN

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn= nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc= nn.Linear(hidden_size,1)
    def forward(self,x):
        h0= torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _= self.rnn(x,h0)
        out=self.fc(out[:, -1, :])
        return out


In [84]:
input_size= X_train.shape[1]
model= RNN(input_size)
criterion= nn.BCELoss()
optimizer=optim.Adam(model.parameters())

### Traning the RNN

In [86]:
epochs=10
for epoch in range(epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad() 
        Xb= Xb.unsqueeze(1)
        outputs= model(Xb)
        outputs=torch.sigmoid(outputs.squeeze())
        loss=criterion(outputs, yb)
        loss.backward()
        loss.backward()
        optimizer.step()
    print(f"epoch ={epoch+1}/{epochs} and loss ={loss.item()}")


NotImplementedError: Module [RNN] is missing the required "forward" function

In [ ]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs=model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals+=yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy = {correct_vals/tot_vals*100}")